In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import joblib

In [2]:
df = pd.read_csv("train.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
print("Dataset Shape:", df.shape)

df.info()

Dataset Shape: (891, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [4]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

df[["SibSp", "Parch", "FamilySize", "IsAlone"]].head()

,SibSp,Parch,FamilySize,IsAlone
0,1,0,2,0
1,1,0,2,0
2,0,0,1,1
3,1,0,2,0
4,0,0,1,1


In [5]:
X = df.drop("Survived", axis=1)
y = df["Survived"]

In [6]:
X = X.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1)

In [7]:
numerical_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone"
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked"
]

In [8]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (712, 9)
Testing data: (179, 9)


In [11]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

In [12]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'SibSp', 'Parch',
                                                   'Fare', 'FamilySize',
                                                   'IsAlone']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Pclass', 'Sex',
                                                   'Embarked'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [13]:
y_pred = pipeline.predict(X_test)

In [14]:
pipeline_accuracy = accuracy_score(y_test, y_pred)

print(f"Pipeline Accuracy: {pipeline_accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Pipeline Accuracy: 81.56%

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.90      0.86       110
           1       0.81      0.68      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.82      0.82      0.81       179



In [15]:
X_original = df.drop("Survived", axis=1)

X_original = X_original.drop(
    ["PassengerId", "Name", "Ticket", "Cabin", "FamilySize", "IsAlone"],
    axis=1
)

y_original = df["Survived"]

In [16]:
original_numerical_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

original_categorical_features = [
    "Pclass",
    "Sex",
    "Embarked"
]

In [17]:
original_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

original_categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [18]:
original_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            original_numeric_transformer,
            original_numerical_features
        ),
        (
            "cat",
            original_categorical_transformer,
            original_categorical_features
        )
    ]
)

In [19]:
original_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            original_numeric_transformer,
            original_numerical_features
        ),
        (
            "cat",
            original_categorical_transformer,
            original_categorical_features
        )
    ]
)

In [20]:
original_pipeline = Pipeline(
    steps=[
        ("preprocessor", original_preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

In [21]:
X_train_original, X_test_original, y_train_original, y_test_original = train_test_split(
    X_original,
    y_original,
    test_size=0.2,
    random_state=42,
    stratify=y_original
)

In [22]:
original_pipeline.fit(
    X_train_original,
    y_train_original
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'SibSp', 'Parch',
                                                   'Fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Pclass', 'Sex',
                                                   'Embarked'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [23]:
original_pred = original_pipeline.predict(X_test_original)

original_accuracy = accuracy_score(
    y_test_original,
    original_pred
)

print(f"Original Pipeline Accuracy: {original_accuracy:.2%}")

Original Pipeline Accuracy: 80.45%


In [24]:
comparison = pd.DataFrame({
    "Model": [
        "Without Engineered Features",
        "With Engineered Features"
    ],
    "Accuracy": [
        original_accuracy,
        pipeline_accuracy
    ]
})

comparison["Accuracy"] = comparison["Accuracy"].round(4)

comparison

,Model,Accuracy
0,Without Engineered Features,0.8045
1,With Engineered Features,0.8156


## Feature Engineering Results

Two new features, FamilySize and IsAlone, were created from the existing SibSp and Parch columns. FamilySize represents the total number of family members travelling with a passenger, while IsAlone identifies passengers travelling without family members. The model was evaluated both with and without these engineered features. The comparison shows whether adding these features improved the model's ability to predict passenger survival.

## What is an ML Pipeline?

An ML pipeline combines data preprocessing and model training into one reusable workflow. It ensures that the same preprocessing steps are applied consistently to both training and testing data. This helps prevent data leakage and makes machine learning code cleaner, safer, and easier to reproduce.

In [25]:
joblib.dump(pipeline, "titanic_survival_pipeline.joblib")

print("Pipeline saved successfully!")

Pipeline saved successfully!


## Conclusion

The Titanic survival prediction model was converted into a complete Scikit-learn pipeline using ColumnTransformer. Numerical features were handled with median imputation and StandardScaler, while categorical features were processed using imputation and OneHotEncoder. Two engineered features, FamilySize and IsAlone, were added to provide additional information about passengers' travelling circumstances. The model with engineered features was compared with the original feature set to determine whether feature engineering improved prediction performance. The final trained pipeline was saved using joblib for future use.